# 11 · 티처블 머신 모델을 코랩에서 쓰기 — 재활용 분류 프로그램
교과서 **131~133쪽**(컴퓨터 비전을 활용한 쓰레기 분리 배출 · 재활용 분류 프로그램)을 우리 수업 방식으로 옮겼습니다.

**순서**
1. [티처블 머신](https://teachablemachine.withgoogle.com/train/image) → 이미지 프로젝트 → 클래스 `플라스틱 · 유리 · 종이`에 사진 넣기 → 모델 학습
2. **모델 내보내기 → Tensorflow 탭 → Keras → 모델 다운로드** → zip 안의 `keras_model.h5` 와 `labels.txt`
3. 이 노트북에서 두 파일과 분류할 사진을 올리기

> 사진은 각 클래스를 **비슷한 개수**로, 밝기 · 각도 · 배경을 **다양하게**. 한쪽으로 쏠리면 모델도 쏠립니다(데이터 편향).

## ① 모델 파일 올리기 (`keras_model.h5`, `labels.txt` 두 개를 함께 선택)

In [ ]:
!pip -q install tf_keras
import tf_keras
from google.colab import files
files.upload()
# 티처블 머신 파일에 있는 옛 설정값(groups)을 새 판이 못 읽는 문제를 피하는 줄
class DW(tf_keras.layers.DepthwiseConv2D):
    def __init__(self, **kw):
        kw.pop('groups', None); super().__init__(**kw)
model = tf_keras.models.load_model('keras_model.h5', compile=False, custom_objects={'DepthwiseConv2D': DW})
names = [l.strip().split(' ', 1)[-1] for l in open('labels.txt', encoding='utf-8')]
print(names)

## ② 사진 올려서 분류하기
티처블 머신 모델은 **224×224 크기**, 픽셀 값을 **−1~1** 로 바꿔 넣어야 합니다.
💡 가장 큰 확률의 번호를 찾는 함수 → `np.argmax`

In [ ]:
import numpy as np
from PIL import Image, ImageOps
up = files.upload(); fname = list(up.keys())[0]
img = ImageOps.fit(Image.open(fname).convert('RGB'), (224, 224))
x = (np.asarray(img, dtype=np.float32) / 127.5 - 1)[None]
p = model.predict(x)[0]
for n, v in zip(names, p): print(f'{n:8s} {v:.3f}')
print('→ 이 사진은', names[____(p)], '(으)로 추정됩니다')
img

## ③ 생각해 보기 (교과서 132쪽 «모델 개선하기»)
- 틀리게 분류한 사진은 어떤 특징이 있나요? (각도 · 빛 · 배경 · 라벨 오류)
- 그 경우의 사진을 **더 모아 다시 학습**하면 나아지나요? 바꾼 것과 결과를 한 줄로 적어 보세요.
- 티처블 머신은 이미 많은 사진으로 학습한 모델에 우리 사진을 «조금 더» 가르칩니다 — 이것을 **전이 학습**이라고 합니다(AI 모델 도감 참고).

> 적어 보기:

---
### 정답 예시 (막혔을 때만)
- ② `names[np.argmax(p)]`
- `load_model` 에서 오류가 나면 ① 셀의 `tf_keras` 설치 줄을 먼저 실행했는지 확인하세요(새 텐서플로와 티처블 머신 형식의 차이 때문).